<a href="https://colab.research.google.com/github/magesh786/agri/blob/main/hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install torch torchvision datasets transforms

In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader



In [3]:
class VGGResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, downsample=False):
        super().__init__()
        stride = 2 if downsample else 1

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        self.skip = None
        if downsample or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        if self.skip is not None:
            identity = self.skip(x)

        out += identity
        return self.relu(out)


In [4]:
class VGGResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, downsample=True)
        self.layer3 = self._make_layer(128, 256, 2, downsample=True)
        self.layer4 = self._make_layer(256, 512, 2, downsample=True)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_ch, out_ch, blocks, downsample=False):
        layers = [VGGResBlock(in_ch, out_ch, downsample)]
        for _ in range(1, blocks):
            layers.append(VGGResBlock(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


In [5]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [6]:
from PIL import Image
import numpy as np
import os
import shutil

def create_dummy_images_for_classes(base_directory, class_names, num_images_per_class=5, size=(224, 224)):
    print(f"Creating dummy images in: {base_directory}")
    for class_name in class_names:
        class_dir = os.path.join(base_directory, class_name)
        os.makedirs(class_dir, exist_ok=True)

        for i in range(num_images_per_class):
            filename = f"image_{class_name}_{i:03d}.jpg"
            filepath = os.path.join(class_dir, filename)
            if not os.path.exists(filepath):
                img = Image.fromarray(np.random.randint(0, 255, size + (3,), dtype=np.uint8))
                img.save(filepath)
                # print(f"Created dummy image: {filepath}")

        if num_images_per_class == 0 and not os.listdir(class_dir):
            print(f"Warning: No images created for class '{class_name}' and directory is empty: {class_dir}")

train_base_dir = "/content/sample_data/imagetrain/train"
test_base_dir = "/content/sample_data/imagetrain/test"

# Ensure base directories exist and clear any old 'images' subfolders
for d in [train_base_dir, test_base_dir]:
    os.makedirs(d, exist_ok=True)
    old_images_folder = os.path.join(d, 'images')
    if os.path.exists(old_images_folder):
        shutil.rmtree(old_images_folder)
        print(f"Removed old 'images' folder: {old_images_folder}")
    # Also remove any stray files in the base directory that might interfere with ImageFolder
    for item in os.listdir(d):
        item_path = os.path.join(d, item)
        if os.path.isfile(item_path) and item.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
            os.remove(item_path)
            print(f"Removed stray image file: {item_path}")


# Define our new binary classes
class_names = ['yes', 'no']

print("Ensuring dummy images are present in training directory...")
create_dummy_images_for_classes(train_base_dir, class_names, num_images_per_class=10) # Create 10 images per class for training

print("Ensuring dummy images are present in testing directory...")
create_dummy_images_for_classes(test_base_dir, class_names, num_images_per_class=3) # Create 3 images per class for testing

print("Dummy image creation for binary classification complete. Re-running the dataset setup.")


Ensuring dummy images are present in training directory...
Creating dummy images in: /content/sample_data/imagetrain/train
Ensuring dummy images are present in testing directory...
Creating dummy images in: /content/sample_data/imagetrain/test
Dummy image creation for binary classification complete. Re-running the dataset setup.


In [7]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
import shutil

# Define VGGResBlock class (from cell pnJCw6Cg9M2S)
class VGGResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, downsample=False):
        super().__init__()
        stride = 2 if downsample else 1

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        self.skip = None
        if downsample or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        if self.skip is not None:
            identity = self.skip(x)

        out += identity
        return self.relu(out)

# Define VGGResNet class (from cell uOBraPUH9Qn2)
class VGGResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, downsample=True)
        self.layer3 = self._make_layer(128, 256, 2, downsample=True)
        self.layer4 = self._make_layer(256, 512, 2, downsample=True)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_ch, out_ch, blocks, downsample=False):
        layers = [VGGResBlock(in_ch, out_ch, downsample)]
        for _ in range(1, blocks):
            layers.append(VGGResBlock(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# Define transforms (from cell V64voKJv9e3f)
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Data Loaders (assuming 'yes' and 'no' subfolders are created by the dummy data script)
train_dataset = datasets.ImageFolder("/content/sample_data/imagetrain/train", transform=train_transform)
test_dataset  = datasets.ImageFolder("/content/sample_data/imagetrain/test", transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Now num_classes will correctly be 2 (for 'yes' and 'no')
num_classes = len(train_dataset.classes)
print("Classes:", train_dataset.classes)

# Instantiate model, criterion, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VGGResNet(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Model, criterion, and optimizer initialized.")


Classes: ['no', 'yes']
Model, criterion, and optimizer initialized.


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VGGResNet(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [9]:
import os

!ls -R /content/sample_data/imagetrain/train

/content/sample_data/imagetrain/train:
no  yes

/content/sample_data/imagetrain/train/no:
image_no_000.jpg  image_no_003.jpg  image_no_006.jpg  image_no_009.jpg
image_no_001.jpg  image_no_004.jpg  image_no_007.jpg
image_no_002.jpg  image_no_005.jpg  image_no_008.jpg

/content/sample_data/imagetrain/train/yes:
image_yes_000.jpg  image_yes_003.jpg  image_yes_006.jpg  image_yes_009.jpg
image_yes_001.jpg  image_yes_004.jpg  image_yes_007.jpg
image_yes_002.jpg  image_yes_005.jpg  image_yes_008.jpg


In [ ]:
epochs = 10

train_losses = []
val_losses = []

for epoch in range(epochs):
    model.train()
    running_train_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Validation phase
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(test_loader)
    val_losses.append(avg_val_loss)

    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")


Epoch [1/10] Train Loss: 0.7799, Val Loss: 0.6940
Epoch [2/10] Train Loss: 0.6273, Val Loss: 0.7441
Epoch [3/10] Train Loss: 1.0411, Val Loss: 1.9400
Epoch [4/10] Train Loss: 2.7772, Val Loss: 1.8521


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")


In [ ]:
torch.save(model.state_dict(), "vgg_resnet_hybrid.pth")
model.load_state_dict(torch.load("vgg_resnet_hybrid.pth"))
model.eval()


In [ ]:
def check_disease(image_path):
    image = Image.open(image_path).convert("RGB")
    image = test_transform(image)
    image = image.unsqueeze(0).to(device)  # Add batch dimension

    with torch.no_grad():
        output = model(image)
        probs = torch.softmax(output, dim=1)
        confidence, predicted_class = torch.max(probs, 1)

    class_names = train_dataset.classes
    disease_status = class_names[predicted_class.item()]

    return disease_status, confidence.item()


In [ ]:
image_path = "/content/sample_data/OIP (1).webp"

result, confidence = check_disease(image_path)

print("Diagnosis:", result)
print(f"Confidence: {confidence*100:.2f}%")


In [ ]:
image_path = "/content/normal.webp"

result, confidence = check_disease(image_path)

print("Diagnosis:", result)
print(f"Confidence: {confidence*100:.2f}%")